# Case Study 8.2 - Generating a Dataset for Idiom Translation

In the second case study of chapter 8 we explore using multi-lingual language models to generate a custom dataset to train models that classify different patterns of idiom translation. The goal is to build a tool to help editors within a publishing company focus on book translations that require their attention.

The raw python scripts containing the book listsings can be found in the files:

* [CaseStudy_8.2_01.py](CaseStudy_8.2_01.py)
* [CaseStudy_8.2_02.py](CaseStudy_8.2_02.py)
* [CaseStudy_8.2_03.py](CaseStudy_8.2_03.py)
* [CaseStudy_8.2_04.py](CaseStudy_8.2_04.py)
* [CaseStudy_8.2_05.py](CaseStudy_8.2_05.py)
  

## 01 - Create Idioms

In [ ]:
import re
import json
import itertools
from tqdm import tqdm
import pandas as pd
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def get_list_from_file(file):
    filename = f"data/{file}"
    with open(filename, "r") as f:
        my_list = [line.strip() for line in f.readlines()]
    return my_list

In [ ]:
prompt = PromptTemplate(
   template = """
     You are an expert on the English language.
     You will generate a single sentence expression of an idea
     that is consistent with the following criteria.
     ---
     Topic: It should relate to {TOPIC}
     Words: Make use of {CASE}s.
     Style: It should have a {STYLE} style.
     ---
     Generate a single sentence of text, nothing else.
     """,
   input_variables=[ "TOPIC", "CASE", "STYLE"],
)

In [ ]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="llama3.2")

In [ ]:
chain = prompt | model | StrOutputParser()

In [ ]:
cases = ['idiom', 'no idiom']
styles = ['colloquial', 'literary']

topics_file = "topics.txt"
topics = get_list_from_file(topics_file)

In [ ]:
results = pd.DataFrame()

combinations = list(itertools.product(topics, cases))

for row in tqdm(combinations, desc=f"Processing file in: {topics_file}"):
   t = row[0]
   c = row[1]
   for s in styles:
      record = {"TOPIC": t, "CASE":c, "STYLE":s}
      try:
          response = chain.invoke(record)
          record['text'] = response

      except Exception as e:
          record['text'] = "FAILURE"
      results = pd.concat([results, pd.DataFrame([record])], ignore_index=True)

results.to_csv("data/generated_expressions.csv", index_label='ID')

## 02 - Audit Model

We then run a second model to eliminate bad data generated in the first run

In [ ]:
import re
import json
import pandas as pd
from tqdm import tqdm
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.output_parsers import JsonOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
model_name = "gemini-2.5-pro"
model = ChatGoogleGenerativeAI(model=model_name)

dataset = "data/generated_expressions.csv"
df = pd.read_csv(dataset)

In [ ]:
class Audit(BaseModel):
    has_idiom: bool = Field(
        description="A Boolean value that indicates if the sentence\
                     provided contains an idiom."
    )

parser = JsonOutputParser(pydantic_object=Audit)

In [ ]:
prompt = PromptTemplate(
   template = """
     You are an expert on the English language.
     Your job is to determine whether the sentence
     below contains an English idiom.
     ---
     Sentence: {SENTENCE}
     ---
     {format_instructions}
     """,
   input_variables=[ "SENTENCE"],
   partial_variables={
      "format_instructions":parser.get_format_instructions()
   }
)

In [ ]:
chain = prompt | model | parser

results = pd.DataFrame()

records = [x for i,x in df.iterrows()]

In [ ]:
for r in tqdm(records, desc=f"Processing file in: {dataset}"):
         record = {"SENTENCE": r['text']}
         try:
             response = chain.invoke(record)
             r['has_idiom'] = response['has_idiom']
         except Exception as e:
             print("Error processing record", e)
             r['has_idiom'] = np.nan
         results = pd.concat([results, pd.DataFrame([r])], ignore_index=True)

results.to_csv("data/audited_expressions.csv")

## 03 - Manual Audit

We then take a look at the results of the auditing model to ensure that it is improving the results and not adding noise.

The following cell contains a small script that was used for the manual audit.
You can also just run it in a terminal as follows:
```
uv run CaseStudy_8.2_03.py
```

In [ ]:
import pandas as pd

df = pd.read_csv("data/audited_expressions.csv")

smp = df.sample(frac=1, random_state=42)

print("########### Audit these examples ########")
model1 = 0
model2 = 0
bothwrong = 0
samples = 2
for i in range(samples):
    print(f" GENERATOR: {smp.iloc[i]['CASE']}")
    print(f" AUDIT HAS IDIOM: {smp.iloc[i]['has_idiom']}")
    print(smp.iloc[i]['text'])
    print("----------------------------")
    valid1 = input("CASE RIGHT (y/n)")
    if valid1 == "y":
        model1 += 1
    valid2 = input("AUDIT RIGHT (y/n)")
    if valid2 == "y":
        model2 += 1
    if (valid1 == "n") & (valid2 == "n"):
        bothwrong += 1
    print("##############################################")

print("----------------------------------------")
print(f"Audit results")
print(f"Generating Model: {str(model1)} / {samples} Correct")
print(f"  Auditing Model: {str(model2)} / {samples} Correct")
print("----------------------------------------")
print(f" Both Models: {str(bothwrong)} / {samples} Errors")


## 04 - Filter

Based on that audit we determined that the quality of the data significantly improved so we used it to filter the data as follows

In [ ]:
import pandas as pd

df = pd.read_csv("data/audited_expressions.csv")


filtered = df[ ((df['CASE']=='idiom') & (df['has_idiom']==True)) |
               ((df['CASE']=='no idiom') & (df['has_idiom']==False))
]

filtered.to_csv("data/audited_expressions_filtered.csv", index=False)

filtered_records = len(df)-len(filtered)
pct = round(100*filtered_records/len(df), 2)

print(f"Filtered {pct}% Records down to {len(filtered)} Total Records")


## 05 - Translate

We then apply a translation to the idiom data to produce alternative translations as targets for our detection model

In [ ]:
import re
import json
import pandas as pd
from tqdm import tqdm
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

In [ ]:
model = ChatOllama(model="qwen3:30b")

target_language = "Spanish"

instructions = {
   "idiom":{
      "IdiomLiteral": "Ignore the idiom in this sentence and translate\
                       it perfectly literally",
      "idiomPositive": f"Identify the idiom in this sentence and translate\
                       it into semantically equivalent {target_language}"
   },
   "no idiom":{
      "NoIdiomLiteral": f"Translate this sentence into {target_language}\
                          without using idioms",
      "NoIdiomPositive": f"Translate this sentence into {target_language}\
                          using an appropriate idiom",
   }
}


In [ ]:
prompt = PromptTemplate(
   template = """
     You are an expert translator working between English and {TARGET}.
     We give you a sentence in English below which you should translate
     into {TARGET}. You should use the additional instruction below to
     determine how you do the translation.
     ---
     Instruction: {INSTRUCTION}
     Sentence: {SENTENCE}
     ___
     Generate a single sentence transation, nothing else
     """,
   input_variables=["TARGET", "INSTRUCTION", "SENTENCE"],
)

chain = prompt | model | StrOutputParser()

In [ ]:
dataset = "data/audited_expressions_filtered.csv"
df = pd.read_csv(dataset)

results = pd.DataFrame()

records = [x for i,x in df.iterrows()]

In [ ]:
for r in tqdm(records, desc=f"Processing file in: {dataset}"):

         cased = r['CASE']
         instr = instructions[cased]
         for ins in instr.keys():
             record = {"TARGET": target_language, "SENTENCE": r['text']}
             record["INSTRUCTION"] = instr[ins]
             try:
                 response = chain.invoke(record)
                 r['instruction'] = ins
                 r['translation'] = response
             except Exception as e:
                 print("Error processing record", e)
                 r['instruction'] = ins
                 r['translation'] = np.nan
             results = pd.concat([results, pd.DataFrame([r])], ignore_index=True)

results.to_csv("data/translated_expressions.csv")